# Verify a bibliography and surface hallucinated citations in a manuscript

A paper's reference list is its evidence of dialogue with existing knowledge. Traditionally, tools like Zotero let us **collect, organize, format** citations to journal style, assuming every entry genuinely exists. But when a draft originates from (or passes through) a large language model, this assumption breaks—the list fills with **hallucinated citations**: flawless DOI syntax, convincing titles, yet pairing a real paper's identity with wrong authors, or missing DOI entirely. Naked human review almost cannot catch these; they wound scholarly integrity most, yet Zotero only manages citations without judging veracity, and CrossRef only exposes metadata when you query a DOI.

So the core of bibliography verification is not formatting—it is **judging each entry true or false**. Can its DOI resolve? Is the record complete? Most critically—do the authors match the true paper behind the title? Judging truth requires a trusted **ground-truth source**, and single sources have blind spots, so we triangulate across three sources (local known table / resolver / online Crossref/OpenAlex) with **three-library verification**, and the first hit wins. This tutorial's main course: a complete pipeline from search to verification to manuscript audit, catching hallucinated citations before they enter the final text.

We walk the full chain using `socialverse`'s `sv.lit` (literature) toolset: search screening (`search_free`) → personal library fan-out (`zotero_bridge`) → **three-library verification (`verify_citations`)** → citation styling (`citation_manage`) → literature landscape (`literature_map`) → manuscript audit (`manuscript_review`). Data comes from a built-in toy bibliography `ds.load_bib()`: only 4 records, but deliberately seeded with one suspicious entry lacking a DOI (`sus1`) and one chimeric hallucination that grafts Braun & Clarke's real paper onto Foucault (`chi1`)—perfectly showing each step's judgment. Functionally, this chain parallels Zotero (personal library + CSL style) plus CrossRef / OpenAlex (DOI metadata resolution) plus journal desk review.

## Setup

Pin matplotlib to the Agg backend (headless, kernel/CI-safe), then import socialverse. Save all figures to the notebook directory so relative paths `![](fig.png)` work from any working directory. We also provide a minimal `display()` fallback so this notebook runs as a plain script without error.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 必须在 pyplot 被任何地方 import 之前设定

import base64
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

# 图保存到 notebook 同目录(与 ![](fig.png) 相对引用对齐)。
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # 交互式 Jupyter 里没有 __file__
    _HERE = os.getcwd()


def _fig(name):
    return os.path.join(_HERE, name)


try:  # 真正的 Jupyter 里用富显示;当普通脚本跑时回退到 print
    from IPython.display import display
except Exception:  # pragma: no cover
    def display(obj):
        print(obj)


pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 120)

print("socialverse version:", sv.__version__)

socialverse version: 0.1.0


## Load the bibliography

`ds.load_bib()` returns a raw bibliography of 4 records. The first two (`ok1` / `ok2`) are real qualitative-research classics. `sus1` is a **suspicious record**—appears complete but lacks a DOI for resolution. `chi1` is a **hallucinated citation**—it reuses `ok2` (Braun & Clarke's *Using thematic analysis in psychology*) true title and true DOI, but lists the author as Foucault with year 1975. This type of fabrication—true paper identity plus wrong author—is exactly what verification targets. Here are the raw 4 records:

In [2]:
raw_bib = ds.load_bib()
pd.DataFrame(raw_bib)[["id", "title", "authors", "year", "doi"]]

,id,title,authors,year,doi
0,ok1,The Practice of Reflexivity in Qualitative Research,"[Finlay, L.]",2002,10.1177/104973202129120052
1,ok2,Using thematic analysis in psychology,"[Braun, V., Clarke, V.]",2006,10.1191/1478088706qp063oa
2,sus1,A Framework for Digital Distant Reading,"[Nobody, A.]",2021,None
3,chi1,Using thematic analysis in psychology,"[Foucault, M.]",1975,10.1191/1478088706qp063oa


## Search screening: normalize to unified schema

First step: normalize the records to a single format and deduplicate. `search_free` defaults offline (connects only if `online=True` is passed and `requests` is available; otherwise never connects), consumes our `records=[...]`, organizes each into unified schema `id/title/authors/year/doi/...` written to `sources['bib']`, and retains a deduplication ledger for manual screening (include/exclude/pending).

In [3]:
st = sv.StudyState()
sv.lit.search_free(st, records=raw_bib, query="thematic analysis reflexivity")

print("规范化后条数:", len(st.sources["bib"]))
print("保留的 id  :", [r["id"] for r in st.sources["bib"]])

规范化后条数: 3
保留的 id  : ['ok1', 'ok2', 'sus1']


**Note: `chi1` is gone.** `search_free` deduplicates by DOI first, else title; the hallucinated record `chi1` reused `ok2`'s true DOI (`10.1191/1478088706qp063oa`), so it was merged as a duplicate and discarded. This is a real trap: deduplication is good, but it hides hallucinated citations that hijack true DOIs at the screening stage. Remember—genuine verification must bypass dedup and submit the **original 4 records** to the truth table to surface `chi1`. This also shows why curation and verification are two separate operations.

The deduplication ledger records the search pattern, whether it connected, the candidate count; one row per entry, `screen='pending'`, ready for downstream manual screening:

In [4]:
cit_ledger = st.evidence["citations"]
print("检索台账:mode=%s  online=%s  n_candidates=%s  query=%r" % (
    cit_ledger["mode"], cit_ledger["online"],
    cit_ledger["n_candidates"], cit_ledger["query"]))
pd.DataFrame(cit_ledger["candidates"])[["id", "title", "year", "doi", "screen"]]

检索台账:mode=records  online=False  n_candidates=3  query='thematic analysis reflexivity'


,id,title,year,doi,screen
0,ok1,The Practice of Reflexivity in Qualitative Research,2002,10.1177/104973202129120052,pending
1,ok2,Using thematic analysis in psychology,2006,10.1191/1478088706qp063oa,pending
2,sus1,A Framework for Digital Distant Reading,2021,,pending


## Personal library fan-out: score and rank by relevance

This step parallels Zotero library search: `zotero_bridge` scores each record in `sources['bib']` against the query using **five strategies** (title / author / tags / fulltext / notes), fuses the scores into a relevance `_score`, deduplicates, and sorts descending by score. Without a Zotero connection, it degrades to pure in-process text matching—fully offline, deterministic. We search for `thematic analysis` and see which strategies each record matched and its fusion score:

In [5]:
sv.lit.zotero_bridge(st, query="thematic analysis")
ranked = st.sources["bib"]
pd.DataFrame([{
    "id": r["id"],
    "title": r["title"][:44],
    "_score": round(r["_score"], 3),
    "matched": ",".join(r["_matched_strategies"]),
} for r in ranked])

,id,title,_score,matched
0,ok2,Using thematic analysis in psychology,0.469,"fulltext,title"
1,sus1,A Framework for Digital Distant Reading,0.078,title
2,ok1,The Practice of Reflexivity in Qualitative R,0.064,title


`ok2` (*Using thematic analysis in psychology*) scores highest, hitting title and fulltext as expected. But note: after `search_free` dedup, only 3 records remain in the pool; the hallucinated record `chi1` was already discarded, so no sorting recovers it. **Ranking alone does not resolve truth**—this is precisely why the next verification step exists.

## Three-library verification: surface chimeric and suspicious

This is the centerpiece. `verify_citations` sorts each citation into one of four completeness states:

- **`verified`** — DOI syntax valid, bibliographic record complete, no contradictory ground truth.
- **`suspicious`** — Record otherwise complete but DOI missing or malformed; cannot be parsed for verification.
- **`not_found`** — Neither valid DOI nor matching ground truth found; no evidence.
- **`chimeric`** — Title matches known truth highly, but author sets do not align (author Jaccard overlap below `author_cut`): a true paper's identity grafted onto wrong authors—the signature hallucinated citation.

Ground truth comes from three sources, prioritized **resolver > local known table > online Crossref/OpenAlex**, with three-way triangulation where the first hit wins (this is the literal meaning of three-library verification). Fully offline by default; only `online=True` connects, and network failure silently downgrades to offline judgment without error.

To surface `chi1`, we do two things: (a) bypass the deduplicated pool and write **original 4 records** directly to `sources['bib']`; (b) supply a **local known table `known`**—the authoritative author identities for those two DOIs as CrossRef or a trusted reference source would provide them. This table is the "second library" in the triangle.

In [6]:
# (a) 绕过去重:把原始 4 条(含 chi1)直接登记为待核验的 bib
st_verify = sv.StudyState()
st_verify.write("sources", "bib", raw_bib)

# (b) 本地已知真值表(= CrossRef/受信库对这两个 DOI 给出的真实作者)
known = [
    {"title": "Using thematic analysis in psychology",
     "authors": ["Braun, V.", "Clarke, V."], "doi": "10.1191/1478088706qp063oa"},
    {"title": "The Practice of Reflexivity in Qualitative Research",
     "authors": ["Finlay, L."], "doi": "10.1177/104973202129120052"},
]

sv.lit.verify_citations(st_verify, known=known)
vb = st_verify.evidence["verified_bib"]

print("核验汇总:")
print(json.dumps(vb["summary"], ensure_ascii=False, indent=2))
print("\n状态计数:", vb["tally"])

核验汇总:
{
  "n_records": 4,
  "n_verified": 2,
  "n_suspicious": 1,
  "n_not_found": 0,
  "n_chimeric": 1,
  "pass_rate": 0.5,
  "flagged_ids": [
    "sus1",
    "chi1"
  ],
  "online_used": false,
  "ground_truth": "known",
  "note": "结合本地/在线真值三角核验"
}

状态计数: {'verified': 2, 'suspicious': 1, 'not_found': 0, 'chimeric': 1}


The summary shows `n_chimeric=1`, `ground_truth='known'`, `flagged_ids=['sus1','chi1']`—both problematic citations caught. Row by row, see the judgment reason and which library resolved it (`resolved_via`):

In [7]:
pd.DataFrame([{
    "id": r["id"],
    "status": r["status"],
    "via": r["resolved_via"],
    "doi_valid": r["doi_valid"],
    "reason": r["reason"],
} for r in vb["records"]])

,id,status,via,doi_valid,reason
0,ok1,verified,known,True,"三角核验命中已知文献(标题 sim=1.00,作者一致)"
1,ok2,verified,known,True,"三角核验命中已知文献(标题 sim=1.00,作者一致)"
2,sus1,suspicious,offline,False,"题录完整但缺少有效 DOI,无法解析核验"
3,chi1,chimeric,known,True,标题与已知文献匹配(sim=1.00)但作者不符(overlap=0.00<0.34):疑似张冠李戴的杜撰引文


Reading this table is an evidence audit:

- `ok1` / `ok2` → **verified** (via `known`): title similarity 1.00 and author match; triangulation hits ground truth.
- `sus1` → **suspicious** (via `offline`): record complete but lacks valid DOI; offline rules also cannot pass.
- `chi1` → **chimeric** (via `known`): title vs. true Braun & Clarke paper similarity 1.00, but author set `{foucault}` vs. truth `{braun, clarke}` overlap 0.00, below threshold 0.34—citation with misattributed identity flagged. This is the error hardest to spot by eye in LLM drafts, yet most damaging to integrity.

Why is the `known` table essential? Because **rules only check syntax; ground truth checks identity**. Without the `known` table, relying solely on offline rules, `chi1` would be marked `verified` because the DOI syntax is flawless and the record complete—hallucination escapes. Run both modes side by side and the difference is stark:

In [8]:
st_rules = sv.StudyState()
st_rules.write("sources", "bib", raw_bib)
sv.lit.verify_citations(st_rules)  # 纯规则,无 known 表
tally_rules = st_rules.evidence["verified_bib"]["tally"]
chi_status_rules = next(r["status"] for r in st_rules.evidence["verified_bib"]["records"]
                        if r["id"] == "chi1")
print("纯离线规则     tally:", tally_rules, "| chi1 →", chi_status_rules, "(幻觉逃逸!)")
print("加本地 known 表 tally:", vb["tally"], "| chi1 →", "chimeric", "(被揪出)")

纯离线规则     tally: {'verified': 3, 'suspicious': 1, 'not_found': 0, 'chimeric': 0} | chi1 → verified (幻觉逃逸!)
加本地 known 表 tally: {'verified': 2, 'suspicious': 1, 'not_found': 0, 'chimeric': 1} | chi1 → chimeric (被揪出)


Plot the verification results as a status-composition bar chart to see at a glance that "only half of 4 records truly pass":

In [9]:
order = ["verified", "suspicious", "not_found", "chimeric"]
colors = {"verified": "#2e7d32", "suspicious": "#ef6c00",
          "not_found": "#757575", "chimeric": "#c62828"}
counts = [vb["tally"].get(s, 0) for s in order]

fig, ax = plt.subplots(figsize=(6.4, 3.6))
bars = ax.bar(order, counts, color=[colors[s] for s in order],
              edgecolor="black", linewidth=0.6)
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.03, str(c),
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylabel("# references")
ax.set_ylim(0, max(counts) + 0.6)
# 图内文字用英文,避免默认 DejaVu 字体缺中文字形的 tofu(叙事仍在 markdown 里用中文)
ax.set_title("verify_citations - reference integrity (n=4)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(_fig("fig_verify_status.png"), dpi=150, bbox_inches="tight")  # PNG → tight
plt.close(fig)
print("saved:", _fig("fig_verify_status.png"))

saved: /Users/fernandozeng/Desktop/analysis/omicos-project/socialverse-worktrees/tutorials-rewrite/notebooks/fig_verify_status.png


![Verification status composition](fig_verify_status.png)

## Citation styling: format for target journal

With the verified and suspicious citations in hand, `citation_manage` formats them to target journal style—APA 7 / Vancouver / Nature are real template implementations, not placeholders. APA sorts by author surname; numeric style preserves citation order. This parallels Zotero's CSL citation-style engine. We generate APA and Vancouver styles for the 3 deduplicated records (verified + suspicious):

In [10]:
sv.lit.citation_manage(st, style="APA")
apa = st.evidence["citations"]
print("=== APA 7 (n=%d) ===" % apa["n"])
for f in apa["formatted"]:
    print(f"  [{f['n']}] {f['reference']}")

sv.lit.citation_manage(st, style="Vancouver")
van = st.evidence["citations"]
print("\n=== Vancouver (n=%d) ===" % van["n"])
for f in van["formatted"]:
    print(f"  [{f['n']}] {f['reference']}")

=== APA 7 (n=3) ===
  [1] Braun, V., & Clarke, V. (2006). Using thematic analysis in psychology. https://doi.org/10.1191/1478088706qp063oa
  [2] Finlay, L. (2002). The Practice of Reflexivity in Qualitative Research. https://doi.org/10.1177/104973202129120052
  [3] Nobody, A. (2021). A Framework for Digital Distant Reading.

=== Vancouver (n=3) ===
  [1] Braun V, Clarke V. Using thematic analysis in psychology. 2006. doi:10.1191/1478088706qp063oa
  [2] Nobody A. A Framework for Digital Distant Reading. 2021.
  [3] Finlay L. The Practice of Reflexivity in Qualitative Research. 2002. doi:10.1177/104973202129120052


`artifacts['tables']` also retains a row-by-row DataFrame (n / id / formatted / style) for direct table insertion or export:

In [11]:
st.artifacts["tables"]

,n,id,formatted,style
0,1,ok2,"Braun V, Clarke V. Using thematic analysis in psychology. 2006. do...",Vancouver
1,2,sus1,Nobody A. A Framework for Digital Distant Reading. 2021.,Vancouver
2,3,ok1,Finlay L. The Practice of Reflexivity in Qualitative Research. 200...,Vancouver


## Literature landscape: schools, landmark works, temporal arc

`literature_map` reads the bib, builds a keyword co-occurrence graph (proxy for co-citation), uses `networkx` greedy modularity for community detection to identify **schools of thought**; then computes **landmark works** (ranked by influence proxy), debate axes, and **temporal arc**. This step is not predictive; it is methodological knowledge-landscape mapping. Our corpus is only 3 records, so schools degenerate to single-node clusters, but the mechanism is identical to real large corpora:

In [12]:
sv.lit.literature_map(st)
land = st.evidence["landscape"]
print("n_references=%d  n_schools=%d" % (land["n_references"], land["n_schools"]))
print("cluster_method:", land["cluster_method"])
print("\n代表作(按影响力代理排序):")
for w in land["seminal_works"]:
    print("  · %s (%s) — %s [influence=%.2f]" % (
        ", ".join(w["authors"]) or "?", w["year"], w["title"][:48], w["influence"]))
print("\n时间脉络:")
for t in land["timeline"]:
    print("  %d  n=%d  · %s" % (t["year"], t["n"], t["exemplar"][:52]))

n_references=3  n_schools=3
cluster_method: networkx greedy-modularity(keyword co-occurrence)

代表作(按影响力代理排序):
  · Finlay, L. (2002) — The Practice of Reflexivity in Qualitative Resea [influence=3.98]
  · Braun, V., Clarke, V. (2006) — Using thematic analysis in psychology [influence=2.94]
  · Nobody, A. (2021) — A Framework for Digital Distant Reading [influence=2.79]

时间脉络:
  2002  n=1  · The Practice of Reflexivity in Qualitative Research
  2006  n=1  · Using thematic analysis in psychology
  2021  n=1  · A Framework for Digital Distant Reading


When matplotlib is available, `literature_map` renders a knowledge-landscape quadrant scatter (x = temporal midpoint, y = school size) as base64 data-URI in `artifacts['figures']` (kernel-safe, returnable). We decode it and save to the notebook directory for markdown reference via `![](fig.png)`:

In [13]:
figs = st.artifacts.get("figures") or {}
land_fig = figs.get("landscape")
if land_fig and land_fig.get("data_uri"):
    _, b64 = land_fig["data_uri"].split(",", 1)
    with open(_fig("fig_landscape.png"), "wb") as fh:
        fh.write(base64.b64decode(b64))
    print("saved:", _fig("fig_landscape.png"), "·", land_fig.get("caption"))
else:
    print("matplotlib 不可用,literature_map 已 fail-soft 跳过图(结构化 landscape 仍在)")

saved: /Users/fernandozeng/Desktop/analysis/omicos-project/socialverse-worktrees/tutorials-rewrite/notebooks/fig_landscape.png · 文献知识地形图(象限:时间×流派规模)


![Literature knowledge landscape](fig_landscape.png)

## Manuscript audit: claim→evidence support and readiness decision

This closes the chain. We verified the bibliography; now we take manuscript text and check whether **each claim in the text has support from a verified citation**. `manuscript_review` does four things:

1. Use regex to extract in-text citations (APA parenthetical / narrative / numeric);
2. Reconcile with `evidence['verified_bib']`—identify `orphan` citations (cited but not in verified_bib) and `uncited` entries (in verified_bib but never cited in text);
3. Audit each sentence for claim→evidence support; scan for causal/absolute phrasing (cause / prove / always / every…) and check for mismatches; label each as `supported / unsupported / over-claim`;
4. Synthesize `supported_ratio` and issue readiness decision `BLOCKER / MAJOR / MINOR / READY`.

This step takes `evidence['verified_bib']` as input, so we reuse the verification result from `st_verify` above, then feed in a deliberately mined manuscript: 1 sentence with normal citation, 1 with narrative citation, 1 with causal/absolute phrasing + orphan citation, 1 with causal/absolute phrasing + no citation.

In [14]:
manuscript = (
    "Thematic analysis is a widely used qualitative method (Braun & Clarke, 2006). "
    "Reflexivity strengthens credibility, as Finlay (2002) argues. "
    "Our data prove definitively that autonomy causes engagement in every case (Smith, 2019). "
    "The results show a significant effect on burnout across all teams."
)
# 稿件正文登记到 sources['datasets'](manuscript_review 的输入之一)
st_verify.write("sources", "datasets", manuscript)

sv.lit.manuscript_review(st_verify, manuscript=manuscript)
ce = st_verify.evidence["claim_evidence"]

pd.DataFrame([{
    "claim": c["claim_id"],
    "status": c["status"],
    "cited": c["cited"],
    "causal": c["causal_language"],
    "hedged": c["hedged"],
    "sentence": c["sentence"][:60],
} for c in ce["claims"]])

,claim,status,cited,causal,hedged,sentence
0,C1,supported,True,False,False,Thematic analysis is a widely used qualitative method (Braun
1,C2,supported,True,False,False,"Reflexivity strengthens credibility, as Finlay (2002) argues"
2,C3,over-claim,True,True,False,Our data prove definitively that autonomy causes engagement
3,C4,over-claim,False,True,False,The results show a significant effect on burnout across all


Reading the table: C1 / C2 have verified-citation support → `supported`; C3 "prove definitively … causes … every case" is causal/absolute phrasing, cites `(Smith, 2019)` which is not in verified_bib → `over-claim`; C4 "significant effect … across all teams" is likewise causal/absolute with no citation → `over-claim`.

Citation reconciliation also flags two classes of issues: `(Smith, 2019)` is an **orphan citation** (verified_bib has no entry) and `foucault` / `nobody` are **uncited entries** (in verified_bib but never cited in text). This consistency audit is exactly what Zotero will not do for you:

In [15]:
print("引注配平 balance:")
print(json.dumps(ce["balance"], ensure_ascii=False, indent=2))

引注配平 balance:
{
  "n_citations": 3,
  "orphans": [
    "(Smith, 2019)"
  ],
  "uncited_reference_ids": [
    "chi1",
    "ok1",
    "ok2",
    "sus1"
  ],
  "uncited_authors": [
    "foucault",
    "nobody"
  ]
}


Finally, check `diagnostics['coverage']` for readiness: `supported_ratio=0.5`, 2 over-claims, 1 orphan citation; decision **MAJOR** (any over-claim or orphan triggers at least MAJOR; exceeding threshold escalates to BLOCKER). This is a sentence ready for a reviewer comment: "Manuscript not ready—half of claims lack support from verified citations, over-causal assertions and orphan citations present; revise before resubmission."

In [16]:
cov = st_verify.diagnostics["coverage"]
print("就绪裁决:")
print(json.dumps(cov, ensure_ascii=False, indent=2))

就绪裁决:
{
  "n_claims": 4,
  "n_supported": 2,
  "n_unsupported": 0,
  "n_overclaim": 2,
  "supported_ratio": 0.5,
  "n_orphan_citations": 1,
  "n_uncited_references": 2,
  "verdict": "MAJOR",
  "note": "就绪裁决:BLOCKER/MAJOR/MINOR/READY"
}


In [17]:
st_verify.artifacts["tables"]

,claim_id,type,severity,excerpt,hint
0,C3,over-claim,MAJOR,Our data prove definitively that autonomy causes engagement in eve...,因果/绝对措辞缺乏支撑或对冲
1,C4,over-claim,MAJOR,The results show a significant effect on burnout across all teams.,因果/绝对措辞缺乏支撑或对冲
2,None,orphan_citation,MAJOR,"(Smith, 2019)",引注未在 verified_bib 中找到匹配
3,None,uncited_reference,MINOR,foucault,已核验参考文献从未被正文引用
4,None,uncited_reference,MINOR,nobody,已核验参考文献从未被正文引用


## Reproducible evidence chain

Unlike a standard verification script, `socialverse` adds one thing. Before calling each core function above, it checks whether required upstream slots are ready—e.g., `manuscript_review` requires `evidence['verified_bib']` to exist; if missing, it errors immediately and tells you which step to run first, rather than silently returning a wrong conclusion. After successful execution, what this step consumed and produced is automatically logged in a read-only provenance ledger. So `st.summary()` at the chain's end is not just a snapshot of "which slots were filled," but an auditable evidence chain: who wrote which slot under what contract, at a glance. This is especially vital for literature verification—"which library and rule flagged this citation as hallucinated" matters as much as the verdict itself.

In [18]:
print(st_verify.summary())
print()
print("provenance 账本(每一步的契约):")
for p in st_verify.provenance:
    fn = p["function"].split(".")[-1]
    req = sv.utils._fmt_slots(p["requires"]) or "∅"
    pro = sv.utils._fmt_slots(p["produces"]) or "∅"
    print("  step %d · %-18s  requires[%s]  →  produces[%s]" % (
        p["step"], fn, req, pro))

StudyState
  sources: bib, datasets
  diagnostics: coverage
  evidence: verified_bib, claim_evidence
  artifacts: tables
  provenance: 2 step(s)

provenance 账本(每一步的契约):
  step 1 · verify_citations    requires[sources['bib']]  →  produces[evidence['verified_bib']]
  step 2 · manuscript_review   requires[sources['datasets'], evidence['verified_bib']]  →  produces[evidence['claim_evidence'], diagnostics['coverage'], artifacts['tables']]


## Summary

We completed a full literature chain: search screening → personal library fan-out → **three-library verification** → citation styling → literature landscape → manuscript audit. Functionally it parallels Zotero (personal library + CSL style) plus CrossRef / OpenAlex (DOI metadata resolution) plus journal desk review.

But against these point-and-click tools, we add two critical things: first, `verify_citations` implements three-library triangulation with local known table / resolver / online source, judges each citation for truth on sight, and surfaces the misattributed hallucinations (`chi1`: real title, real DOI, wrong author) that Zotero cannot export and CrossRef will not volunteer. Second, each step automatically accumulates into an auditable evidence chain, turning bibliography work from a black-box point-and-click operation into a traceable analysis pipeline. The next tutorial [10_full_study_evidence_chain](10_full_study_evidence_chain.ipynb) chains these methods into a fully reproducible mini-study and exports the entire evidence chain as a deliverable artifact.